# Background pools for the presence-vs-background models

**Purpose.** The only place where the two background pools are built:
the uniform pool shared by every model of the effort-reduction curve
and the forest-only pool of its robustness check. It also writes
`data/processed_manifest.csv`, which records what each file in
`data/processed/` was built from (hash, inputs, seed, parameters).

**Why here.** A derived file must be rebuilt when its inputs change
(`CLAUDE.md`, "Derived files"). The modelling notebooks of
`07_foundation_models/` do not build the pools: they read them with
`src.background.load_background`, which stops if the file or one of its
inputs no longer matches the manifest.

**How a pool is rebuilt.** The points are sampled again from the
current inputs (presences, expert polygons, pilot extent, seed). If
they are the stored ones, the stored embeddings are kept and only the
manifest row is updated. If they differ, new TESSERA embeddings are
needed: that is a remote read and runs only with `ALLOW_FETCH = True`.

What the pools are and why (buffer, size, seeds):
[`data/README.md`](../../data/README.md).

In [ ]:
import sys
from pathlib import Path

import pandas as pd

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
sys.path.insert(0, str(ROOT))
from src import manifest
from src.background import ensure_pool, paths

THIS = "notebooks/05_data_integration/background_pools.ipynb"
ALLOW_FETCH = False  # True only after deciding to read new TESSERA embeddings

## 1. Pilot embeddings

Read on 2026-10-20 by a notebook that no longer exists (see
`data/README.md`), so their inputs are not known. Each embedding
belongs to a pixel and does not depend on which presences are kept;
the rows record the hash of each file as it is now.

In [ ]:
known = manifest.read(ROOT)
added = 0
for block in sorted(paths(ROOT)["pilot"].glob("block_*.npz")):
    if block.relative_to(ROOT).as_posix() not in known:
        manifest.record(
            ROOT, block,
            built_by="RQ1 pilot notebook removed on 2026-10-20 (see data/README.md)",
            inputs=manifest.UNKNOWN_INPUTS,
            params={"year": 2023, "tessera": "GeoTessera v1.1 Zarr (GeoTesseraZarr.sample_points), geotessera 0.11.0"},
        )
        added += 1
print(f"pilot blocks: {added} rows added, {len(list(paths(ROOT)['pilot'].glob('block_*.npz')))} files")

## 2. Uniform pool

2 000 points, seed 42, 50 m exclusion buffer around every
citizen-science presence and expert polygon.

In [ ]:
row = ensure_pool(ROOT, "uniform", built_by=THIS,
                  built_by_if_kept="notebooks/07_foundation_models/one_class_method_selection.ipynb",
                  allow_fetch=ALLOW_FETCH)
print(row["status"], row["file"], row["sha256"][:16], "built on", row["built_on"])

## 3. Forest-only pool

2 000 points, seed 0, same buffer, kept only in a COS 2023 forest or
shrubland class.

In [ ]:
row = ensure_pool(ROOT, "forest_only", built_by=THIS,
                  built_by_if_kept="notebooks/07_foundation_models/forest_only_background_check.ipynb",
                  allow_fetch=ALLOW_FETCH)
print(row["status"], row["file"], row["sha256"][:16], "built on", row["built_on"])

## 4. Manifest

In [ ]:
m = pd.read_csv(ROOT / manifest.MANIFEST)
print(len(m), "rows")
for _, r in m[m["file"].str.contains("background")].iterrows():
    print()
    print(r["file"])
    print("  sha256:", r["sha256"])
    print("  built :", r["built_on"], "by", r["built_by"])
    print("  seed  :", r["seed"], "| params:", r["params"])
    for ref in r["inputs"].split(";"):
        print("  input :", ref[: ref.rfind("@") + 17])